In [1]:
import pandas as pd

USEFUL_COLUMNS = ['Name', 'Sex', 'Age', 'BodyweightKg', 'Equipment',
                   'Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg',
                   'TotalKg', 'Date']

power_data = pd.read_csv("data/openpowerlifting.csv", usecols = USEFUL_COLUMNS)


In [2]:
power_data['Date'] = pd.to_datetime(power_data['Date'])


In [3]:
power_data = power_data[power_data['Equipment'] == 'Raw'].copy()


In [4]:
power_data = power_data.dropna(subset=['Age', 'BodyweightKg']).copy()


In [5]:
power_data = power_data.sort_values(['Name', 'Date'])

power_data = power_data[power_data['Sex'] != 'Mx'].copy()
power_data['Sex_encoded'] = power_data['Sex'].map({'M': 0, 'F': 1})

In [6]:
squat_data = power_data[power_data['Best3SquatKg'] > 0].copy()
squat_data = squat_data.sort_values(['Name', 'Date'])
squat_data['experience'] = squat_data.groupby('Name').cumcount() + 1

squat_data[['Name', 'Date', 'Best3SquatKg', 'Sex_encoded', 'experience']].head(10)

,Name,Date,Best3SquatKg,Sex_encoded,experience
1400040,A Aanvi,2026-05-15,77.5,1,1
289339,A Ajeesha,2017-12-04,112.5,1,1
282960,A Ashwin,2012-12-10,170.0,0,1
2482449,A Belousov,2019-10-01,75.0,0,1
1413739,A Hemanth Kumar,2022-08-10,175.0,0,1
152513,A I Temaki,2023-12-17,375.0,0,1
484698,A I Temaki,2024-04-13,370.0,0,2
485081,A I Temaki,2025-03-29,400.0,0,3
2809445,A Jay Montanez,2016-02-27,225.0,0,1
1398505,A K S Shri Ram,2019-09-26,117.5,0,1


In [7]:
bench_data = power_data[power_data['Best3BenchKg'] > 0].copy()
bench_data = bench_data.sort_values(['Name', 'Date'])
bench_data['experience'] = bench_data.groupby('Name').cumcount() + 1

bench_data[['Name', 'Date', 'Best3BenchKg', 'Sex_encoded', 'experience']].head(10)

,Name,Date,Best3BenchKg,Sex_encoded,experience
1400040,A Aanvi,2026-05-15,40.0,1,1
289339,A Ajeesha,2017-12-04,55.0,1,1
1402214,A Arun Kumar,2018-11-14,135.0,0,1
282960,A Ashwin,2012-12-10,95.0,0,1
2482449,A Belousov,2019-10-01,75.0,0,1
3732954,A Cavatorta,2004-06-19,110.0,0,1
1413739,A Hemanth Kumar,2022-08-10,105.0,0,1
484698,A I Temaki,2024-04-13,212.0,0,1
485081,A I Temaki,2025-03-29,220.0,0,2
331027,A J Ceronio,2016-12-04,55.0,0,1


In [8]:
deadlift_data = power_data[power_data['Best3DeadliftKg'] > 0].copy()
deadlift_data = deadlift_data.sort_values(['Name', 'Date'])
deadlift_data['experience'] = deadlift_data.groupby('Name').cumcount() + 1

deadlift_data[['Name', 'Date', 'Best3DeadliftKg', 'Sex_encoded', 'experience']].head(10)

,Name,Date,Best3DeadliftKg,Sex_encoded,experience
1400040,A Aanvi,2026-05-15,90.0,1,1
289339,A Ajeesha,2017-12-04,132.5,1,1
282960,A Ashwin,2012-12-10,220.0,0,1
2482449,A Belousov,2019-10-01,100.0,0,1
1413739,A Hemanth Kumar,2022-08-10,200.0,0,1
484698,A I Temaki,2024-04-13,235.0,0,1
485081,A I Temaki,2025-03-29,235.0,0,2
331027,A J Ceronio,2016-12-04,130.0,0,1
2809445,A Jay Montanez,2016-02-27,270.0,0,1
1398505,A K S Shri Ram,2019-09-26,150.0,0,1


In [9]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def train_lift_model(data, target_column):
    FEATURES = ['Age', 'BodyweightKg', 'Sex_encoded', 'experience']
    X = data[FEATURES]
    y = data[target_column]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = Ridge()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    naive_prediction = y_train.mean()
    naive_mae = mean_absolute_error(y_test, [naive_prediction] * len(y_test))

    print(f"{target_column} — Naive MAE: {naive_mae:.2f}, Model MAE: {mae:.2f}, Model RMSE: {rmse:.2f}")

    return model

In [10]:
squat_model = train_lift_model(squat_data, 'Best3SquatKg')
bench_model = train_lift_model(bench_data, 'Best3BenchKg')
deadlift_model = train_lift_model(deadlift_data, 'Best3DeadliftKg')

Best3SquatKg — Naive MAE: 48.65, Model MAE: 27.24, Model RMSE: 35.86
Best3BenchKg — Naive MAE: 38.25, Model MAE: 20.50, Model RMSE: 26.93
Best3DeadliftKg — Naive MAE: 50.47, Model MAE: 28.92, Model RMSE: 38.07


In [11]:
import joblib

joblib.dump(squat_model, 'squat_population_model.joblib')
joblib.dump(bench_model, 'bench_population_model.joblib')
joblib.dump(deadlift_model, 'deadlift_population_model.joblib')

print("Saved")

Saved


In [12]:
def build_progression_rates(lift_column):
    df = power_data[['Name', 'Date', lift_column]].dropna(subset=[lift_column]).copy()
    df = df.sort_values(['Name', 'Date'])

    df['experience'] = df.groupby('Name').cumcount() + 1

    df['prev_lift'] = df.groupby('Name')[lift_column].shift(1)
    df['prev_date'] = df.groupby('Name')['Date'].shift(1)

    df['delta_kg'] = df[lift_column] - df['prev_lift']
    df['delta_days'] = (df['Date'] - df['prev_date']).dt.days

    df = df.dropna(subset=['delta_kg', 'delta_days'])
    df = df[df['delta_days'] > 0]

    df['rate_per_day'] = df['delta_kg'] / df['delta_days']
    return df

squat_rates = build_progression_rates('Best3SquatKg')
print(squat_rates[['Name', 'experience', 'delta_kg', 'delta_days', 'rate_per_day']].head())
print(squat_rates.groupby('experience')['rate_per_day'].median())

               Name  experience  delta_kg  delta_days  rate_per_day
484698   A I Temaki           2      -5.0       118.0     -0.042373
485081   A I Temaki           3      30.0       350.0      0.085714
1405300   A Madhuri           2       5.0       425.0      0.011765
1408641   A Madhuri           3      -2.5       364.0     -0.006868
1400125   A Madhuri           4      20.0       708.0      0.028249
experience
2      0.040984
3      0.031056
4      0.025381
5      0.021552
6      0.019841
         ...   
96     0.625000
97    -0.727273
98     0.930233
100   -0.185185
101   -0.032468
Name: rate_per_day, Length: 84, dtype: float64


In [13]:
summary = squat_rates.groupby('experience')['rate_per_day'].agg(['median', 'count'])
print(summary)

              median   count
experience                  
2           0.040984  168223
3           0.031056  110442
4           0.025381   72023
5           0.021552   51838
6           0.019841   36518
...              ...     ...
96          0.625000       1
97         -0.727273       1
98          0.930233       1
100        -0.185185       1
101        -0.032468       1

[84 rows x 2 columns]


In [14]:
def bucket_experience(exp):
    if exp <= 5:
        return '1-5'
    elif exp <= 10:
        return '6-10'
    elif exp <= 20:
        return '11-20'
    elif exp <= 40:
        return '21-40'
    else:
        return '41+'

squat_rates['experience_bucket'] = squat_rates['experience'].apply(bucket_experience)
bucket_summary = squat_rates.groupby('experience_bucket')['rate_per_day'].agg(['median', 'count'])
print(bucket_summary)


                     median   count
experience_bucket                  
1-5                0.032468  402526
11-20              0.010373   41707
21-40              0.003497    6669
41+                0.000000     336
6-10               0.017094  111650


In [15]:
bench_rates = build_progression_rates('Best3BenchKg')
bench_rates['experience_bucket'] = bench_rates['experience'].apply(bucket_experience)
bench_bucket_summary = bench_rates.groupby('experience_bucket')['rate_per_day'].agg(['median', 'count'])
print(bench_bucket_summary)

                     median   count
experience_bucket                  
1-5                0.016190  541229
11-20              0.000000   86386
21-40              0.000000   23592
41+                0.000000    2987
6-10               0.007215  176220


In [16]:
bucket_summary = squat_rates.groupby('experience_bucket')['rate_per_day'].agg(['median', 'var', 'count'])
print(bucket_summary)

bench_bucket_summary = bench_rates.groupby('experience_bucket')['rate_per_day'].agg(['median', 'var', 'count'])
print(bench_bucket_summary)

                     median       var   count
experience_bucket                            
1-5                0.032468  1.289652  402526
11-20              0.010373  8.526419   41707
21-40              0.003497  8.663513    6669
41+                0.000000  0.126471     336
6-10               0.017094  2.054512  111650
                     median        var   count
experience_bucket                             
1-5                0.016190   0.532288  541229
11-20              0.000000   3.245305   86386
21-40              0.000000  10.906004   23592
41+                0.000000   9.456615    2987
6-10               0.007215   1.583747  176220
